# Employment Data ETL Pipeline

In [ ]:
# !pip install sqlalchemy
# !pip install psycopg2

In [ ]:
import pandas as pd

# FUNCTION: Read & Filter a CSV in chunks
def load_filtered_chunks(
    file_path,
    usecols,
    filter_qtr,
    filter_disclosure,
    chunksize=100_000
):
    """
    Reads a CSV in chunks, keeps only selected columns,
    and filters rows keeping 2nd quarter data where 
    disclosure_code is null.
    """
    for chunk in pd.read_csv(
        file_path,
        usecols=usecols,
        chunksize=chunksize,
        low_memory=False
    ):
        # Filter rows
        chunk = chunk[chunk[filter_qtr] == 2]
        chunk = chunk[chunk[filter_disclosure].isna()]

         # Normalize column names to lowercase to match postgresql
        chunk.columns = [col.lower() for col in chunk.columns]

        yield chunk

In [ ]:
from sqlalchemy import create_engine

# FUNCTION: Upload a chunk to PostgreSQL
def upload_chunk(df, table_name, engine):
    df.to_sql(
        table_name,
        engine,
        if_exists="append",
        index=False,
        method="multi",
        chunksize=10_000
    )

In [3]:
# FUNCTION: Process a file
def process_file(file_path, usecols, filter_qtr, filter_disclosure, table_name, engine):
    for chunk in load_filtered_chunks(file_path, usecols, filter_qtr, filter_disclosure):
        upload_chunk(chunk, table_name, engine)


In [1]:
import pandas as pd

# ESTABLISH THE POSTGRESQL CONNECTION

import glob
from sqlalchemy import create_engine

# PostgreSQL connection parameters
from utility import portnumber, host, database, username, password

# Connection parameters
HOST = host
PORT = portnumber
DBNAME = database
USER = username
PASSWORD = password

# Ensure Server is running and Destination Table exists within the database
engine = create_engine(f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DBNAME}")


C:\Users\KDSharp\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


### QCEW NAICS-Based Data - Single File - Quarterly<br />
https://www.bls.gov/cew/downloadable-data-files.htm

In [ ]:
# PROCESS EACH FILE IN THE DIRECTORY
files = glob.glob("data/employment/quarterly/*.csv") 

for file in files:
    process_file(
        file_path=file,
        usecols= [          # Extract columns of interest to match SQL Table. E.g.Month 3 employment level
            'area_fips',
            'own_code',
            'industry_code',
            'agglvl_code',
            'size_code',
            'year',
            'qtr',
            'disclosure_code',
            'qtrly_estabs',
            'month3_emplvl'
            ],
        filter_qtr="qtr", 
        filter_disclosure="disclosure_code",
        table_name="empl_singlefile_qtrly",
        engine=engine
    )

### Industry Titles

In [3]:
# EXTRACT
empl_industry_titles_df = pd.read_csv("data/employment/industry-titles.csv")

# LOAD
DESTINATION_TABLE = "empl_industry_titles"

# Import Cleaned data to Postgresql destination table
empl_industry_titles_df.to_sql(
    DESTINATION_TABLE,
    engine,
    if_exists="append", # or "replace"
    index=False,
    method="multi"      # batches inserts for speed
)

2678

### Ownership Codes

In [3]:
empl_ownership_titles_df = pd.read_csv("data/employment/ownership-titles-csv.csv")

print(empl_ownership_titles_df.head())

   own_code                 own_title
0         0             Total Covered
1         1        Federal Government
2         2          State Government
3         3          Local Government
4         4  International Government


In [4]:
# EXTRACT
# empl_ownership_titles_df = pd.read_csv("data/employment/ownership-titles-csv.csv")

# LOAD
DESTINATION_TABLE = "empl_ownership_titles"

# Import Cleaned data to Postgresql destination table
empl_ownership_titles_df.to_sql(
    DESTINATION_TABLE,
    engine,
    if_exists="append", # or "replace"
    index=False,
    method="multi"      # batches inserts for speed
)

8